<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/04_week/entropy_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

#Pure Certainty
# [EN] WHY WE WROTE THIS: To demonstrate the absolute lower bound of Shannon Entropy (0.0 Bits).
# [TR] NEDEN YAZDIK: Shannon Entropisinin mutlak alt sınırını (0.0 Bit) görmek ve formülü anlamak için.
# [EN] GOAL: Calculate entropy when a model is 100% confident in a single outcome.
# [TR] AMAÇ: Model tek bir seçenekten %100 emin olduğunda entropinin sıfırlandığını göstermek.

p_certain = [1.0, 0.0]
# Formül / Formula: - sum(P * log2(P))  (0 * log2(0) = 0 kabul edilir)
entropy_1 = -(1.0 * math.log2(1.0) + 0)

print(f"[Code 1] Tam Kesinlik Entropisi / Pure Certainty Entropy: {entropy_1:.4f}")

# [EN] RESULT INSIGHT: 0.0 Bits means absolute certainty. There is zero unpredictability or information chaos.
# [TR] SONUÇ NE SÖYLÜYOR?: 0.0 Bit tam kesinlik demektir. Hiçbir belirsizlik veya bilgi karmaşası yoktur.

[Code 1] Tam Kesinlik Entropisi / Pure Certainty Entropy: -0.0000


In [ ]:
#Pure Uncertainty
# [EN] WHY WE WROTE THIS: To demonstrate maximum binary entropy (1.0 Bit) using an equal probability split.
# [TR] NEDEN YAZDIK: Eşit olasılık dağılımında maksimum ikili entropi değerini (1.0 Bit) göstermek için.
# [EN] GOAL: Measure uncertainty in a coin-flip scenario (50-50 decision split).
# [TR] AMAÇ: %50-%50 yazı-tura senaryosundaki kararsızlık miktarını ölçmek.

p_uncertain = [0.5, 0.5]
entropy_2 = -(0.5 * math.log2(0.5) + 0.5 * math.log2(0.5))

print(f"[Code 2] Tam Kararsızlık Entropisi / Pure Uncertainty Entropy: {entropy_2:.4f}")

# [EN] RESULT INSIGHT: 1.0 Bit is the maximum possible uncertainty for a 2-choice system. The model is totally guessing.
# [TR] SONUÇ NE SÖYLÜYOR?: 1.0 Bit, 2 seçenekli sistemdeki maksimum kararsızlıktır. Model tamamen rastgele tahmin yapmaktadır.

[Code 2] Tam Kararsızlık Entropisi / Pure Uncertainty Entropy: 1.0000


In [ ]:
import torch

# [EN] WHY WE WROTE THIS: Math crash protection. Log2(0) is undefined and produces NaN/Infinity errors in PyTorch.
# [TR] NEDEN YAZDIK: Matematiksel çökme koruması. Log2(0) tanımsızdır ve PyTorch'ta NaN/Sonsuz hatalarına yol açar.
# [EN] GOAL: Add a tiny value (1e-10) to probabilities to make log operations numerically stable.
# [TR] AMAÇ: Olasılıklara çok küçük bir epsilon (1e-10) ekleyerek logaritma işlemini güvenli ve kararlı hale getirmek.

eps = 1e-10
prob_with_zero = torch.tensor([1.0, 0.0, 0.0])
safe_log = torch.log2(prob_with_zero + eps)
entropy_3 = -torch.sum(prob_with_zero * safe_log).item()

print(f"[Code 3] Epsilon Korumalı Entropi / Epsilon-Protected Entropy: {entropy_3:.4f}")

# [EN] RESULT INSIGHT: Safely returns 0.0 instead of crashing code or returning NaN.
# [TR] SONUÇ NE SÖYLÜYOR?: Kodun çökmesini önleyerek güvenli şekilde 0.0 sonucunu verir.

[Code 3] Epsilon Korumalı Entropi / Epsilon-Protected Entropy: -0.0000


In [ ]:
# [EN] WHY WE WROTE THIS: To perform fast tensor calculations without writing manual for-loops.
# [TR] NEDEN YAZDIK: Manuel for döngüleri yazmadan hızlı tensor hesaplamaları yapmak için.
# [EN] GOAL: Calculate multi-class Shannon Entropy using PyTorch vectorized operations.
# [TR] AMAÇ: PyTorch'un vektörel işlemlerini kullanarak çok sınıflı Shannon Entropisini tek satırda hesaplamak.

probs_tensor = torch.tensor([0.70, 0.20, 0.05, 0.05])
entropy_4 = -torch.sum(probs_tensor * torch.log2(probs_tensor + 1e-10)).item()

print(f"[Code 4] Vektörel Tensor Entropisi / Vectorized Tensor Entropy: {entropy_4:.4f}")

# [EN] RESULT INSIGHT: Entropy is 1.18. The model heavily leans on the first option (70%), keeping uncertainty relatively low.
# [TR] SONUÇ NE SÖYLÜYOR?: Entropi 1.18 çıktı. Model %70 ile ilk seçeneğe ağırlık verdiği için kararsızlık göreceli olarak düşüktür.

[Code 4] Vektörel Tensor Entropisi / Vectorized Tensor Entropy: 1.2568


In [ ]:
import torch.nn.functional as F

# [EN] WHY WE WROTE THIS: LLMs produce unnormalized raw scores (logits). We must convert them to percentages.
# [TR] NEDEN YAZDIK: LLM'ler ölçeklenmemiş ham skorlar (logitler) üretir. Bunları yüzdelik oranlara çevirmeliyiz.
# [EN] GOAL: Map raw logit values into a valid probability distribution that sums up to 1.0 (100%).
# [TR] AMAÇ: Ham logit değerlerini toplamı 1.0 (%100) eden geçerli bir olasılık dağılımına dönüştürmek.

sample_logits = torch.tensor([12.5, 3.1, -1.2, 0.5])
sample_probs = F.softmax(sample_logits, dim=-1)

print(f"[Code 5] Softmax Olasılıkları / Softmax Probabilities: {sample_probs.tolist()}")

# [EN] RESULT INSIGHT: The score 12.5 exponentially dominates the rest, receiving almost ~99.9% of the probability weight.
# [TR] SONUÇ NE SÖYLÜYOR?: 12.5 skoru diğerlerini ezerek olasılık ağırlığının neredeyse %99.9'unu tek başına almıştır.

[Code 5] Softmax Olasılıkları / Softmax Probabilities: [0.9999099969863892, 8.271665137726814e-05, 1.1223454521314125e-06, 6.1436594478436746e-06]


In [ ]:
# [EN] WHY WE WROTE THIS: To build a reusable helper function that takes raw model outputs and directly returns entropy.
# [TR] NEDEN YAZDIK: Ham model çıktılarını alıp doğrudan entropi döndüren tekrar kullanılabilir bir yardımcı fonksiyon yazmak için.
# [EN] GOAL: Combine Softmax transformation and Shannon Entropy calculation into a single modular function.
# [TR] AMAÇ: Softmax dönüşümünü ve Shannon Entropisi hesaplamasını modüler tek bir fonksiyonda birleştirmek.

import torch
import torch.nn.functional as F

# [EN] Robust Entropy Function protected against NaN values in large vocabularies
# [TR] Devasa sözlüklerde NaN üretmesini engelleyen güvenli Entropi Fonksiyonu
def logits_to_entropy(logits_vec):
    # 1. Softmax ile olasılıkları hesapla
    p = F.softmax(logits_vec, dim=-1)

    # 2. log2(p) hesapla (Sıfır olan yerler -inf verebilir)
    log_p = torch.log2(p + 1e-12)

    # 3. p * log_p çarpımını yap
    p_log_p = p * log_p

    # 4. KRİTİK ADIM: Oluşan nan veya -inf değerlerini 0.0 ile değiştir
    p_log_p = torch.nan_to_num(p_log_p, nan=0.0, posinf=0.0, neginf=0.0)

    # 5. Entropiyi topla ve negatifini al
    entropy = -torch.sum(p_log_p).item()
    return entropy

entropy_6 = logits_to_entropy(sample_logits)
print(f"[Code 6] Logit'ten Hesaplanan Entropi / Entropy from Logits: {entropy_6:.4f}")

# [EN] RESULT INSIGHT: Very low entropy (~0.008) confirms the model is extremely certain about its top prediction.
# [TR] SONUÇ NE SÖYLÜYOR?: Çok düşük entropi (~0.008), modelin ilk tahmini konusunda son derece kendinden emin olduğunu doğrular.

[Code 6] Logit'ten Hesaplanan Entropi / Entropy from Logits: 0.0014


In [ ]:
# [EN] WHY WE WROTE THIS: To contrast a confident output distribution against a highly hesitant one.
# [TR] NEDEN YAZDIK: Kendinden emin bir çıktı dağılımı ile kararsız/tereddütlü bir dağılımı kıyaslamak için.
# [EN] GOAL: Prove that spread-out logits result in significantly higher entropy values.
# [TR] AMAÇ: Birbirine yakın (yayılmış) logitlerin çok daha yüksek entropi ürettiğini kanıtlamak.

high_conf_logits = torch.tensor([15.0, 1.0, 0.0, -2.0])
low_conf_logits = torch.tensor([4.0, 3.9, 3.8, 3.7])

print(f"[Code 7] Yüksek Güven Entropisi / High Conf Entropy: {logits_to_entropy(high_conf_logits):.4f}")
print(f"[Code 7] Düşük Güven Entropisi / Low Conf Entropy  : {logits_to_entropy(low_conf_logits):.4f}")

# [EN] RESULT INSIGHT: Confident logits yield near-zero entropy (0.0001), while hesitant logits yield high entropy (1.9992).
# [TR] SONUÇ NE SÖYLÜYOR?: Emin logitler sıfıra yakın (0.0001), kararsız logitler ise yüksek entropi (1.9992) verir.

[Code 7] Yüksek Güven Entropisi / High Conf Entropy: 0.0000
[Code 7] Düşük Güven Entropisi / Low Conf Entropy  : 1.9910


In [ ]:
import warnings
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_WARNINGS"] = "1"
import torch
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForCausalLM

# [EN] WHY WE WROTE THIS: To initialize an actual open-source Large Language Model (Qwen2.5) for real-world testing.
# [TR] NEDEN YAZDIK: Gerçek dünya testleri için açık kaynaklı bir Büyük Dil Modelini (Qwen2.5) başlatmak.
# [EN] GOAL: Load model parameters and vocabulary tokenizer onto GPU using Float16 precision for memory efficiency.
# [TR] AMAÇ: Model parametrelerini ve kelime sözlüğünü bellek tasarrufu amacıyla Float16 hassasiyetinde GPU'ya yüklemek.

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")

print(f"[Code 8] Model Başarıyla Yüklendi / Model Successfully Loaded: {model_name}")

# [EN] RESULT INSIGHT: The LLM is ready in GPU VRAM to perform forward pass predictions.
# [TR] SONUÇ NE SÖYLÜYOR?: LLM, ileri yayılım tahminleri yapmak üzere GPU VRAM hafızasında hazırdır.

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Code 8] Model Başarıyla Yüklendi / Model Successfully Loaded: Qwen/Qwen2.5-0.5B


In [ ]:
# [EN] Code 9: Function Extracting Last Token Logits (151,936 Vocab Size)
# [TR] Kod 9: Cümlenin son kelimesinden sonra gelebilecek tüm kelimelerin ham skorlarını çekme.

def get_last_token_logits(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits[0, -1, :] # Son kelimeden sonraki logitler

prompt_fact = "The capital of France is"
last_logits_fact = get_last_token_logits(prompt_fact)

print(f"[Code 9] Çekilen Logit Matris Boyutu / Extracted Logits Shape: {last_logits_fact.shape}")

[Code 9] Çekilen Logit Matris Boyutu / Extracted Logits Shape: torch.Size([151936])


In [ ]:
# [EN] Code 10: Calculating Real Entropy Across the Entire LLM Vocabulary
# [TR] Kod 10: Qwen modelinin 151.936 kelimelik sözlüğünün tamamı üzerinden entropi hesaplama.

real_entropy_fact = logits_to_entropy(last_logits_fact)
print(f"[Code 10] Fransa Başkenti Cümlesi Entropisi / France Capital Entropy: {real_entropy_fact:.4f}")

[Code 10] Fransa Başkenti Cümlesi Entropisi / France Capital Entropy: 4.6758


In [17]:
# [EN] WHY WE WROTE THIS: To demonstrate that factual queries yield lower entropy than open-ended/philosophical prompts.
# [TR] NEDEN YAZDIK: Net bilgi sorularının ucu açık/felsefi cümlelere göre daha düşük entropi ürettiğini göstermek.
# [EN] GOAL: Compare model hesitation between a clear factual prompt and an ambiguous prompt.
# [TR] AMAÇ: Net bir bilgi cümlesi ile belirsiz bir cümle arasındaki tereddüt/kararsızlık farkını ölçmek.

prompt_open = "The meaning of human existence is"
last_logits_open = get_last_token_logits(prompt_open)

entropy_fact = logits_to_entropy(last_logits_fact)
entropy_open = logits_to_entropy(last_logits_open)

print(f"[Code 11] Net Bilgi Entropisi (Fact)       : {entropy_fact:.4f}")
print(f"[Code 11] Ucu Açık Cümle Entropisi (Open) : {entropy_open:.4f}")

# [EN] RESULT INSIGHT: The open-ended prompt produces significantly higher entropy because there are many valid continuation tokens.
# [TR] SONUÇ NE SÖYLÜYOR?: Ucu açık cümle belirgin şekilde daha yüksek entropi üretir çünkü devam edebilecek çok sayıda geçerli kelime vardır.

[Code 11] Net Bilgi Entropisi (Fact)       : 4.6758
[Code 11] Ucu Açık Cümle Entropisi (Open) : 6.8906


In [18]:
# [EN] WHY WE WROTE THIS: To observe LLM behavior when confronted with nonsense/out-of-distribution inputs.
# [TR] NEDEN YAZDIK: Anlamsız/dağılım dışı girdiler verildiğinde LLM'in nasıl davrandığını gözlemlemek için.
# [EN] GOAL: Show that gibberish prompts trigger high uncertainty and elevated entropy scores.
# [TR] AMAÇ: Saçma cümlelerin yüksek kararsızlığı ve tavan yapan entropi skorlarını tetiklediğini göstermek.

prompt_gibberish = "Xyzqkw pdlms blue elephant running upside down in"
entropy_gibberish = logits_to_entropy(get_last_token_logits(prompt_gibberish))

print(f"[Code 12] Saçma Cümle Entropisi / Gibberish Entropy: {entropy_gibberish:.4f}")

# [EN] RESULT INSIGHT: High entropy confirms the model is confused and cannot find a strong statistical continuation pattern.
# [TR] SONUÇ NE SÖYLÜYOR?: Yüksek entropi modelin kafasının karıştığını ve güçlü bir istatistiksel devam kalıbı bulamadığını doğrular.

[Code 12] Saçma Cümle Entropisi / Gibberish Entropy: 6.3438


In [19]:
# [EN] WHY WE WROTE THIS: To analyze how scaling logits with low temperature affects probability sharpness.
# [TR] NEDEN YAZDIK: Logitleri düşük sıcaklıkla ölçeklemenin olasılık keskinliğine etkisini analiz etmek.
# [EN] GOAL: Demonstrate that dividing logits by T < 1.0 artificially suppresses uncertainty and lowers entropy.
# [TR] AMAÇ: Logitleri T < 1.0 değerine bölmenin belirsizliği yapay olarak bastırdığını ve entropiyi düşürdüğünü göstermek.

logits_low_temp = last_logits_fact / 0.2
entropy_low_temp = logits_to_entropy(logits_low_temp)

print(f"[Code 13] Düşük Sıcaklık (T=0.2) Entropisi / Low Temp Entropy: {entropy_low_temp:.4f}")

# [EN] RESULT INSIGHT: Low temperature makes the LLM overly deterministic, collapsing entropy near 0.
# [TR] SONUÇ NE SÖYLÜYOR?: Düşük sıcaklık LLM'i aşırı kuralcı/kararlı hale getirir ve entropiyi 0'a yaklaştırır.

[Code 13] Düşük Sıcaklık (T=0.2) Entropisi / Low Temp Entropy: 0.0631


In [20]:
# [EN] WHY WE WROTE THIS: To analyze how high temperature flattens the output distribution.
# [TR] NEDEN YAZDIK: Yüksek sıcaklığın çıktı dağılımını nasıl düzleştirdiğini analiz etmek.
# [EN] GOAL: Prove that T > 1.0 encourages diversity/randomness, driving entropy upwards.
# [TR] AMAÇ: T > 1.0 ayarının çeşitliliği/rastgeleliği artırarak entropiyi yukarı tırmandırdığını kanıtlamak.

logits_high_temp = last_logits_fact / 2.0
entropy_high_temp = logits_to_entropy(logits_high_temp)

print(f"[Code 14] Yüksek Sıcaklık (T=2.0) Entropisi / High Temp Entropy: {entropy_high_temp:.4f}")

# [EN] RESULT INSIGHT: High temperature boosts randomness, making probabilities more uniform and increasing hallucination risk.
# [TR] SONUÇ NE SÖYLÜYOR?: Yüksek sıcaklık rastgeleliği artırır, olasılıkları homojenleştirir ve halüsinasyon riskini yükseltir.

[Code 14] Yüksek Sıcaklık (T=2.0) Entropisi / High Temp Entropy: 13.4453


In [21]:
# [EN] WHY WE WROTE THIS: Full vocabulary entropy can be noisy due to thousands of near-zero probabilities.
# [TR] NEDEN YAZDIK: Tüm sözlük entropisi, binlerce sıfıra yakın olasılık yüzünden gürültülü sonuçlar verebilir.
# [EN] GOAL: Calculate sub-entropy specifically among the top-K most likely token candidates after re-normalizing.
# [TR] AMAÇ: Yeniden normalize ederek sadece en olası ilk K token adayı arasında "alt entropi" hesaplamak.

def top_k_entropy(logits_vec, k=5):
    probs = F.softmax(logits_vec, dim=-1)
    top_k_probs, _ = torch.topk(probs, k=k)
    top_k_probs_norm = top_k_probs / torch.sum(top_k_probs) # Normalize to sum to 1.0
    return -torch.sum(top_k_probs_norm * torch.log2(top_k_probs_norm + 1e-10)).item()

print(f"[Code 15] Top-5 Alt Entropi (Fact) : {top_k_entropy(last_logits_fact, k=5):.4f}")
print(f"[Code 15] Top-5 Alt Entropi (Open) : {top_k_entropy(last_logits_open, k=5):.4f}")

# [EN] RESULT INSIGHT: Focuses strictly on top contenders, providing a clearer picture of competition between top choices.
# [TR] SONUÇ NE SÖYLÜYOR?: Doğrudan ana adaylara odaklanır ve üst seçenekler arasındaki rekabetin daha net bir resmini sunar.

[Code 15] Top-5 Alt Entropi (Fact) : 1.9004
[Code 15] Top-5 Alt Entropi (Open) : 2.2969


In [22]:
import math

# [EN] WHY WE WROTE THIS: Raw entropy values depend on vocabulary size. Normalization brings values to a universal [0, 1] scale.
# [TR] NEDEN YAZDIK: Ham entropi değerleri sözlük boyutuna bağlıdır. Normalizasyon değerleri evrensel [0, 1] aralığına taşır.
# [EN] GOAL: Divide raw entropy by theoretical maximum entropy (log2(Vocab_Size)).
# [TR] AMAÇ: Ham entropiyi teorik maksimum entropiye (log2(Sözlük_Boyutu)) bölmek.

def normalized_entropy(logits_vec):
    vocab_size = logits_vec.shape[-1]
    max_entropy = math.log2(vocab_size)
    return logits_to_entropy(logits_vec) / max_entropy

print(f"[Code 16] Normalize Entropi / Normalized Entropy: {normalized_entropy(last_logits_fact):.6f}")

# [EN] RESULT INSIGHT: Values closer to 0.0 mean low uncertainty regardless of whether vocabulary size is 30,000 or 150,000.
# [TR] SONUÇ NE SÖYLÜYOR?: Sözlük boyutu ister 30.000 ister 150.000 olsun, 0.0'a yakın değerler evrensel olarak düşük belirsizlik demektir.

[Code 16] Normalize Entropi / Normalized Entropy: 0.271641


In [24]:
# [EN] WHY WE WROTE THIS: To establish an automated rule-based guardrail for detecting potential hallucinations.
# [TR] NEDEN YAZDIK: Potansiyel halüsinasyonları yakalamak için otomatik, kural tabanlı bir güvenlik duvarı kurmak.
# [EN] GOAL: Trigger a warning flag when predicted token entropy crosses a specified threshold.
# [TR] AMAÇ: Tahmin edilen token entropisi belirlenen eşik değerini aştığında bir uyarı bayrağı tetiklemek.

def detect_uncertainty(prompt, threshold=3.5):
    logits = get_last_token_logits(prompt)
    ent = logits_to_entropy(logits)

    print(f"\n[Code 17] Prompt: '{prompt}' | Entropi: {ent:.2f}")
    if ent > threshold:
        print("   ⚠️ UYARI / WARNING: Yüksek Kararsızlık! Halüsinasyon Riski Var!")
    else:
        print("   ✅ GÜVENLİ / SAFE: Model Kendinden Emin.")

detect_uncertainty("The capital of France is", threshold=3.5)
detect_uncertainty("In 2095, the global economy will be driven by", threshold=3.5)

# [EN] RESULT INSIGHT: Acts as a real-time safety monitor flagging speculative or unreliable LLM outputs.
# [TR] SONUÇ NE SÖYLÜYOR?: Spekülatif veya güvenilmez LLM çıktılarını işaretleyen gerçek zamanlı bir güvenlik monitörü görevi görür.


[Code 17] Prompt: 'The capital of France is' | Entropi: 4.68
   ⚠️ UYARI / WARNING: Yüksek Kararsızlık! Halüsinasyon Riski Var!

[Code 17] Prompt: 'In 2095, the global economy will be driven by' | Entropi: 6.46
   ⚠️ UYARI / WARNING: Yüksek Kararsızlık! Halüsinasyon Riski Var!


In [25]:
# [EN] WHY WE WROTE THIS: Uncertainty isn't static; it fluctuates word-by-word as a sentence is generated.
# [TR] NEDEN YAZDIK: Belirsizlik sabit değildir; bir cümle üretilirken kelimeden kelimeye dalgalanır.
# [EN] GOAL: Track step-by-step entropy values across every individual token in a complete prompt sequence.
# [TR] AMAÇ: Tam bir girdi cümlesindeki her bir token boyunca adım adım entropi değerlerini izlemek.

prompt_seq = "Artificial Intelligence will transform human society by"
inputs_seq = tokenizer(prompt_seq, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs_seq = model(**inputs_seq)

all_logits = outputs_seq.logits[0] # [Sequence_Length, Vocab_Size]
tokens = tokenizer.convert_ids_to_tokens(inputs_seq["input_ids"][0])

print("\n[Code 18] Cümle Boyunca Token-by-Token Entropi Analizi:")
for idx, token_str in enumerate(tokens):
    tok_logits = all_logits[idx, :]
    tok_ent = logits_to_entropy(tok_logits)
    print(f"   Token [{idx}] '{token_str.replace('Ġ', '')}': Entropi = {tok_ent:.4f}")

# [EN] RESULT INSIGHT: Pinpoints exact token positions where the LLM experienced confusion during sentence generation.
# [TR] SONUÇ NE SÖYLÜYOR?: Cümle üretimi sırasında LLM'in tam olarak hangi kelime pozisyonlarında kararsızlık yaşadığını tespit eder.


[Code 18] Cümle Boyunca Token-by-Token Entropi Analizi:
   Token [0] 'Art': Entropi = 6.3828
   Token [1] 'ificial': Entropi = 5.3633
   Token [2] 'Intelligence': Entropi = 6.2383
   Token [3] 'will': Entropi = 6.2891
   Token [4] 'transform': Entropi = 5.4961
   Token [5] 'human': Entropi = 5.7812
   Token [6] 'society': Entropi = 4.1055
   Token [7] 'by': Entropi = 6.9062


In [32]:
# [EN] WHY WE WROTE THIS: To wrap all week 4 concepts into an object-oriented, production-ready module.
# [TR] NEDEN YAZDIK: 4. haftanın tüm kavramlarını nesne yönelimli, prodüksiyona hazır modüler bir yapıda toplamak.
# [EN] GOAL: Build an OOP class bundling probability extraction, top-K reporting, and uncertainty thresholding.
# [TR] AMAÇ: Olasılık çekme, Top-K raporlama ve entropi eşik kontrolünü paketleyen bir OOP sınıfı inşa etmek.

class LLMUncertaintyAnalyzer:
    def __init__(self, model_obj, tokenizer_obj, warning_threshold=3.5):
        self.model = model_obj
        self.tokenizer = tokenizer_obj
        self.threshold = warning_threshold

    def analyze(self, prompt_text, top_k=3):
        inputs = self.tokenizer(prompt_text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model(**inputs)

        last_logits = outputs.logits[0, -1, :]
        probs = F.softmax(last_logits, dim=-1)

        entropy_val = -torch.sum(probs * torch.log2(probs + 1e-10)).item()
        top_k_probs, top_k_indices = torch.topk(probs, k=top_k)
        is_risk = entropy_val > self.threshold

        print("\n" + "="*60)
        print(f"📊 ANALİZ RAPORU / ANALYSIS REPORT")
        print(f"Girdi / Input: '{prompt_text}'")
        print(f"Shannon Entropi Skoru: {entropy_val:.4f} Bit")
        print(f"Durum / Status: {'⚠️ YÜKSEK HALÜSİNASYON RİSKİ' if is_risk else '✅ GÜVENLİ TAHMİN'}")
        print("-" * 60)
        print(f"En Yüksek {top_k} Tahmin (Top-{top_k} Candidates):")
        for p, idx in zip(top_k_probs, top_k_indices):
            word = self.tokenizer.decode(idx.item()).strip()
            print(f"   • '{word}': %{p.item()*100:.2f}")
        print("="*60)

# [EN] RESULT INSIGHT: A clean software engine that ingests text and outputs structured reliability metrics.
# [TR] SONUÇ NE SÖYLÜYOR?: Metni alan ve geriye yapılandırılmış güvenilirlik metrikleri döndüren temiz bir yazılım motoru.

In [33]:
# [EN] WHY WE WROTE THIS: To execute end-to-end integration tests using our newly created OOP class.
# [TR] NEDEN YAZDIK: Yeni oluşturduğumuz OOP sınıfını kullanarak uçtan uca entegrasyon testleri çalıştırmak.
# [EN] GOAL: Verify that factual inputs pass safety checks while speculative prompts trigger risk alerts.
# [TR] AMAÇ: Net bilgi girdilerinin güvenlik kontrolünden geçtiğini, spekülatif girdilerin ise risk uyarısı verdiğini doğrulamak.

analyzer = LLMUncertaintyAnalyzer(model, tokenizer, warning_threshold=3.0)

# Test 1: Kesin Bilgi / Factual Prompt
analyzer.analyze("The chemical formula for water is", top_k=3)

# Test 2: Belirsiz / Uncertain Prompt
analyzer.analyze("The winning lottery numbers tomorrow will be", top_k=3)

# [EN] RESULT INSIGHT: Water prompt passes cleanly with low entropy; lottery prompt triggers a high hallucination risk alert.
# [TR] SONUÇ NE SÖYLÜYOR?: Su formülü düşük entropi ile temiz geçer; loto tahmini ise yüksek halüsinasyon riski uyarısını tetikler.


📊 ANALİZ RAPORU / ANALYSIS REPORT
Girdi / Input: 'The chemical formula for water is'
Shannon Entropi Skoru: nan Bit
Durum / Status: ✅ GÜVENLİ TAHMİN
------------------------------------------------------------
En Yüksek 3 Tahmin (Top-3 Candidates):
   • 'H': %56.59
   • '____': %7.20
   • '$\': %5.52

📊 ANALİZ RAPORU / ANALYSIS REPORT
Girdi / Input: 'The winning lottery numbers tomorrow will be'
Shannon Entropi Skoru: nan Bit
Durum / Status: ✅ GÜVENLİ TAHMİN
------------------------------------------------------------
En Yüksek 3 Tahmin (Top-3 Candidates):
   • '': %55.27
   • 'the': %5.65
   • 'a': %5.14
